# Target-Type Classification — Training Pipeline

Trains three classifiers (FFNN, LSTM, SVM) on trajectory data, fuses their
belief distributions, and saves the final test-set output to CSV.

All tuneable values live in **`params.py`** — edit there, not here.

## 1. Imports

In [1]:
%load_ext autoreload
%autoreload 2

from src.config import *
from src.preprocess import *
from src.model_training.models import *
from src.postprocess import *

import params   # central configuration — see params.py

## 2. Load Dataset

In [2]:
data = pd.read_csv(params.INPUT_PATH, header=None)

# Features: columns 0 – 8999  |  Label: second-to-last column
X = data.iloc[:, 0:9000]
y = data.iloc[:, -2]

print(f"Dataset loaded  — X: {X.shape}, y: {y.shape}")
print(f"Unique classes  : {sorted(y.unique())}")

Dataset loaded  — X: (4000, 9000), y: (4000,)
Unique classes  : ['Bomber', 'CivilianCommercialAirliner', 'CivilianHelicopter', 'CruiseMissile', 'MilitaryHelicopter', 'MultiroleFighter', 'SmallCivilian', 'SmallUnmanned']


## 3. Train / Validation / Test Split

In [4]:
train_concat, test_concat, val_concat = split_dataset(
    X, y,
    test_size=params.TYPE_TEST_SIZE,
    val_size=params.TYPE_VAL_SIZE,
)

print(f"Train : {len(train_concat):>6,} samples")
print(f"Val   : {len(val_concat):>6,} samples")
print(f"Test  : {len(test_concat):>6,} samples")

Train :  2,800 samples
Val   :    400 samples
Test  :    800 samples


## 4. Preprocessing

Sliding-window slicing → statistical feature extraction → transformation → scaling → one-hot encoding.

In [5]:
# ── Sliding-window slicing ─────────────────────────────────────────────────
train_slices, test_slices, val_slices = [
    slice_dataset(d, params.K1, params.M, params.W)
    for d in (train_concat, test_concat, val_concat)
]

# ── Statistical Feature Extraction (SFE) ──────────────────────────────────
train_stat, test_stat, val_stat = [
    SFE_dataset(d, params.K1, params.W)
    for d in (train_slices, test_slices, val_slices)
]

# ── Trajectory Transformation ──────────────────────────────────────────────
train_transform, test_transform, val_transform = [
    transform_dataset(d, params.K1, params.W)
    for d in (train_slices, test_slices, val_slices)
]

# ── Feature Scaling ────────────────────────────────────────────────────────
# Scaler-1: fitted on stat features (used by FFNN & SVM)
X_train_1, X_test_1, X_val_1, y_train, y_test, y_val = scale_dataset(
    train_stat, test_stat, val_stat,
    save_path=params.SCALER_STAT_PATH,
    return_labels=True,
)

# Scaler-2: fitted on transformed features (used by LSTM)
X_train_2, X_test_2, X_val_2 = scale_dataset(
    train_transform, test_transform, val_transform,
    save_path=params.SCALER_TRANSFORM_PATH,
    return_labels=False,
)

# ── One-Hot Encoding ───────────────────────────────────────────────────────
y_train_enc, y_test_enc, y_val_enc = OHE_dataset(
    train_stat, test_stat, val_stat,
    save_path=params.OHE_PATH,
)

print("Preprocessing complete.")
print(f"  Stat features shape      : {X_train_1.shape[1]}")
print(f"  Transform features shape : {X_train_2.shape[1]}")

Preprocessing complete.
  Stat features shape      : 99
  Transform features shape : 450


## 5. FFNN Training

In [6]:
ffnn_prob, ffnn_accuracy = ffnn(
    X_train=X_train_1, X_test=X_test_1, X_val=X_val_1,
    y_train=y_train_enc, y_test=y_test_enc, y_val=y_val_enc,
    save_path=params.FFNN_MODEL_PATH,
    num_class=params.NUM_CLASS,
)

print(f"FFNN Accuracy : {ffnn_accuracy:.4f}")

Epoch 1/300
4665/4667 [============================>.] - ETA: 0s - loss: 1.0099 - accuracy: 0.5879
Epoch 1: val_loss improved from inf to 0.79046, saving model to C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\trained_models\ffnn_model.keras
4667/4667 [==============================] - 44s 9ms/step - loss: 1.0099 - accuracy: 0.5879 - val_loss: 0.7905 - val_accuracy: 0.6472
Epoch 2/300
4660/4667 [============================>.] - ETA: 0s - loss: 0.7829 - accuracy: 0.6764
Epoch 2: val_loss improved from 0.79046 to 0.69941, saving model to C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\trained_models\ffnn_model.keras
4667/4667 [==============================] - 41s 9ms/step - loss: 0.7828 - accuracy: 0.6764 - val_loss: 0.6994 - val_accuracy: 0.6985
Epoch 3/300
4664/4667 [============================>.] - ETA: 0s - loss: 0.7098 - accuracy: 0.7053
Epoch 3: val_loss improved from 0.69941 to 0.62545, saving model to C:\Users\tenzi\Desktop\MS_Research_Work\Combat 

## 6. LSTM Training

In [7]:
lstm_prob, lstm_accuracy = lstm(
    X_train=X_train_2, X_test=X_test_2, X_val=X_val_2,
    y_train=y_train_enc, y_test=y_test_enc, y_val=y_val_enc,
    save_path=params.LSTM_MODEL_PATH,
    timesteps=params.W,
    data_dim=params.K1,
    num_class=params.NUM_CLASS,
)

print(f"LSTM Accuracy : {lstm_accuracy:.4f}")

Epoch 1/300
4665/4667 [============================>.] - ETA: 0s - loss: 1.1023 - accuracy: 0.5733
Epoch 1: val_loss improved from inf to 0.90448, saving model to C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\trained_models\lstm_model.keras
4667/4667 [==============================] - 130s 26ms/step - loss: 1.1023 - accuracy: 0.5732 - val_loss: 0.9045 - val_accuracy: 0.6180
Epoch 2/300
4666/4667 [============================>.] - ETA: 0s - loss: 0.8519 - accuracy: 0.6643
Epoch 2: val_loss improved from 0.90448 to 0.83257, saving model to C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\trained_models\lstm_model.keras
4667/4667 [==============================] - 120s 26ms/step - loss: 0.8519 - accuracy: 0.6643 - val_loss: 0.8326 - val_accuracy: 0.6492
Epoch 3/300
4667/4667 [==============================] - ETA: 0s - loss: 0.7866 - accuracy: 0.6895
Epoch 3: val_loss improved from 0.83257 to 0.78730, saving model to C:\Users\tenzi\Desktop\MS_Research_Work\Com

## 7. SVM Training

In [8]:
svm_prob, svm_accuracy = SVM(
    X_train=X_train_1, X_test=X_test_1,
    y_train=y_train,   y_test=y_test,
    save_path=params.SVM_MODEL_PATH,
)

print(f"SVM Accuracy  : {svm_accuracy:.4f}")

SVM Accuracy: 0.7595
SVM Accuracy  : 0.7595


## 8. Persist Accuracies to `params.py`

Writes the just-trained accuracy values back into `params.py` so the
inference notebook (`demo.ipynb`) can import them directly without
re-running training.

In [9]:
import re, pathlib

params_path = pathlib.Path("params.py")
text = params_path.read_text()

def _update(src: str, name: str, value: float) -> str:
    """Replace the `NAME: float = ...` line in params.py with the new value."""
    pattern = rf"^({re.escape(name)}\s*:\s*float\s*=).*$"
    replacement = rf"\1 {value:.6f}"
    return re.sub(pattern, replacement, src, flags=re.MULTILINE)

text = _update(text, "FFNN_ACCURACY", ffnn_accuracy)
text = _update(text, "LSTM_ACCURACY", lstm_accuracy)
text = _update(text, "SVM_ACCURACY",  svm_accuracy)

params_path.write_text(text)

print("params.py updated:")
print(f"  FFNN_ACCURACY = {ffnn_accuracy:.6f}")
print(f"  LSTM_ACCURACY = {lstm_accuracy:.6f}")
print(f"  SVM_ACCURACY  = {svm_accuracy:.6f}")

params.py updated:
  FFNN_ACCURACY = 0.808062
  LSTM_ACCURACY = 0.747063
  SVM_ACCURACY  = 0.759500


## 9. Belief Mapping & Classifier Fusion

In [10]:
# ── Belief mapping ─────────────────────────────────────────────────────────
svm_belief, ffnn_belief, lstm_belief = belief_mapping_type(
    svm_prob,  svm_accuracy,
    ffnn_prob, ffnn_accuracy,
    lstm_prob, lstm_accuracy,
)

# ── Classifier fusion: FFNN ⊕ LSTM → then ⊕ SVM ──────────────────────────
fused_ffnn_lstm     = fuse_ffnn_lstm(ffnn_belief, lstm_belief)
fused_ffnn_lstm_svm = fuse_ffnn_lstm_svm(fused_ffnn_lstm, svm_belief)

print("Belief mapping and classifier fusion complete.")

Belief mapping and classifier fusion complete.


## 10. Temporal Fusion

In [11]:
final_fused = pd.concat(
    [
        pd.DataFrame(temporal_fusion_type(fused_ffnn_lstm_svm)),
        pd.DataFrame(y_test).reset_index(drop=True),
    ],
    axis=1,
)

print(f"Final fused output shape: {final_fused.shape}")
final_fused.head()

Final fused output shape: (16000, 12)


,0,1,2,3,4,5,6,7,8,9,10,Label
0,0.000025,0.000284,0.122716,0.000022,0.010173,0.000052,0.004044,0.757719,0.031277,0.000075,0.073612,SmallUnmanned
1,0.000005,0.000050,0.035787,0.000004,0.001980,0.000009,0.000584,0.948125,0.006900,0.000014,0.006543,SmallUnmanned
2,0.000005,0.000045,0.032259,0.000007,0.002610,0.000013,0.000617,0.948614,0.009170,0.000022,0.006636,SmallUnmanned
3,0.000003,0.000028,0.014455,0.000003,0.001078,0.000005,0.000311,0.976078,0.003509,0.000010,0.004519,SmallUnmanned
4,0.000004,0.000045,0.014506,0.000003,0.001158,0.000006,0.000337,0.975532,0.003781,0.000012,0.004615,SmallUnmanned


## 11. Save Output

In [12]:
import os
os.makedirs(params.OUTPUT_DIR, exist_ok=True)

final_fused.to_csv(params.OUTPUT_CSV, index=False, header=None)

print(f"Saved → {params.OUTPUT_CSV}")

AttributeError: module 'params' has no attribute 'OUTPUT_DIR'